In [7]:
import os

In [ ]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-heart-murmur-detection\\reseach'

In [8]:
os.chdir("../")

In [9]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-heart-murmur-detection'

In [10]:
# 3 entites update
# from config.ymal
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataingestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [13]:
from HeartBeat.constant import *
from HeartBeat.utils.common import read_yaml, create_directories

In [15]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataingestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataingestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [16]:
import os
import urllib.request as request
import zipfile
from HeartBeat.logging import logger
from HeartBeat.utils.common import get_size

In [ ]:
# 5 Components

class DataIngestion:
    def __init__(self, config: DataingestionConfig):
        self.config = config


    def download_files(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} Downloaded with following info \n{headers}")
        else:
            logger.info(f"file already exist of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zipfile(self):
        """
        zip_file_path: str
        Extracts the zip file in dir
        None returns
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [20]:
# 6 Pipeline

try:
    config = configurationManager()
    data_injection_config = config.get_data_ingestion_config()
    data_injection = DataIngestion(config = data_injection_config)
    data_injection.download_files()
    data_injection.extract_zipfile()

except Exception as e:
    raise e

[2026-06-25 17:25:44,135: INFO: common: ymal file config\config.yaml loaded sucessfully]
[2026-06-25 17:25:44,138: INFO: common: ymal file params.yaml loaded sucessfully]
[2026-06-25 17:25:44,143: INFO: common: created directory at artifacts]
[2026-06-25 17:25:44,146: INFO: common: created directory at artifacts/data_injection]
[2026-06-25 17:25:44,149: INFO: 2471477969: file already exist of size: ~ 112595 KB]
